# params-iterable-vs-groups — ex2: polymorphic dispatch handles nn.Parameter and re-iterates safely

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `params-iterable-vs-groups`. Running the final beacon cell reports progress against the `Config: params iterable vs groups` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Config: params iterable vs groups` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`params-iterable-vs-groups`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "params-iterable-vs-groups"
DD_SUBTOPIC = "Config: params iterable vs groups"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## params dispatch — `nn.Parameter` subclass + reused materialization

Ex1 implemented the polymorphic `tensor` vs `dict` dispatch from scratch. The deepening focuses on TWO subtle facets that bite real callers:

1. **`nn.Parameter` IS a `Tensor` subclass** — `isinstance(p, t.Tensor)` is True for any `nn.Parameter`. Code that special-cases `type(first) is t.Tensor` (using `is` instead of `isinstance`) WOULD FAIL when a caller passes `module.parameters()`. Always use `isinstance`.

2. **The dispatch must re-materialize the iterable AFTER peeking.** If you do `first = next(iter(params))` to inspect type, the original iterator has already advanced — the first element is lost on the next pass. Materialize ONCE with `list(params)` up front, then peek at `materialized[0]`. (PyTorch's own source does this.)

**Robust skeleton:**
```python
def normalize(params):
    materialized = list(params)         # one-shot consume
    if not materialized:
        raise ValueError('empty')
    if isinstance(materialized[0], t.Tensor):   # catches nn.Parameter too
        return [{'params': materialized}]
    if isinstance(materialized[0], dict):
        return [dict(g) for g in materialized]
    raise TypeError(...)
```

### Exercise 2 — polymorphic dispatch handles nn.Parameter and re-iterates safely

> ```yaml
> Difficulty: 🔴🔴🔴🔴⚪
> Bloom level: Analyze
> LO: Analyze the `params=` dispatch corner cases so that `nn.Parameter` (a Tensor SUBCLASS) routes through the Tensor branch via `isinstance`, single-use generators are materialized once before peeking, and the returned groups are fresh dicts (no aliasing the caller's input).
> Keywords: polymorphism, isinstance, nn-Parameter, iterator-exhaustion
> ```

**KCs targeted:** `isinstance-vs-type-for-tensor-dispatch`, `materialize-iterator-before-peek`

Implement `ex2_normalize_params_robust(params, default_lr)`.

Same job as ex1's `ex1_normalize_params` but pinned on TWO subtle correctness facets the ex1 spec didn't probe:

1. **`nn.Parameter` MUST route through the Tensor   branch.** Use `isinstance(first, t.Tensor)` (not   `type(first) is t.Tensor`) so that `nn.Parameter`   (a Tensor subclass) is treated as a tensor.
2. **Materialize the iterable BEFORE peeking.** If   `params` is a single-use generator, calling   `next(iter(params))` then iterating again would lose   the first element. Materialize to a list once.
3. **Group dicts in the output are FRESH dicts (shallow   copies) — the caller's input dicts MUST NOT be   mutated.** Adding a fallback `'lr'` key to a   caller-owned dict is a side effect we want to avoid.

Inputs / outputs match ex1: returns `list[dict]` where each dict has at least `'params'` and `'lr'`. Empty input -> `ValueError('empty')`. Wrong element type -> `TypeError`.

In [ ]:
def ex2_normalize_params_robust(params, default_lr: float):
    """Polymorphic params dispatch (ex2 deepening)."""
    raise NotImplementedError()


def _test_ex2():
    # === nn.Parameter is a Tensor subclass -> Tensor branch ===
    p1 = t.nn.Parameter(t.randn(3))
    p2 = t.nn.Parameter(t.randn(2, 4))
    out = ex2_normalize_params_robust([p1, p2], default_lr=1e-3)
    assert isinstance(out, list)
    assert len(out) == 1, 'flat-tensor mode should yield 1 group'
    assert out[0]['lr'] == 1e-3
    assert out[0]['params'][0] is p1
    assert out[0]['params'][1] is p2

    # Make sure the dispatch isn't using `type(...) is t.Tensor` —
    # nn.Parameter has `type() is t.nn.Parameter`, NOT `t.Tensor`.
    assert type(p1) is not t.Tensor, 'sanity: nn.Parameter has its own type'
    assert isinstance(p1, t.Tensor), 'sanity: nn.Parameter is-a Tensor'

    # Module.parameters() generator -> still works.
    mod = t.nn.Linear(4, 6)
    out = ex2_normalize_params_robust(mod.parameters(), default_lr=5e-4)
    assert len(out) == 1
    assert len(out[0]['params']) == 2, 'Linear has weight + bias'
    assert out[0]['lr'] == 5e-4
    # All entries are nn.Parameter instances.
    for p in out[0]['params']:
        assert isinstance(p, t.nn.Parameter)

    # === Single-use generator must be materialized once ===
    gen = (t.nn.Parameter(t.randn(2)) for _ in range(4))
    out = ex2_normalize_params_robust(gen, default_lr=1e-3)
    assert len(out[0]['params']) == 4, (
        f'generator should yield 4 params; got {len(out[0]["params"])}'
    )

    # === Group dicts: missing lr falls through to default_lr ===
    p3 = t.nn.Parameter(t.randn(3))
    p4 = t.nn.Parameter(t.randn(3))
    g1 = {'params': [p3], 'lr': 1e-4}
    g2 = {'params': [p4]}
    out = ex2_normalize_params_robust([g1, g2], default_lr=2e-3)
    assert len(out) == 2
    assert out[0]['lr'] == 1e-4
    assert out[1]['lr'] == 2e-3

    # === Caller's input dicts MUST NOT be mutated ===
    assert 'lr' not in g2, (
        f'g2 should be unchanged; got keys {list(g2.keys())} — '
        'normalize must shallow-copy each group dict, not mutate in place'
    )
    # Output group is a DIFFERENT dict than the input.
    assert out[1] is not g2, 'output group must be a fresh dict'
    # But the 'params' list reference can be shared (no spec on that).

    # === Empty -> ValueError ===
    try:
        ex2_normalize_params_robust([], default_lr=1e-3)
    except ValueError:
        pass
    else:
        raise AssertionError('expected ValueError on empty')

    # === Bad element type -> TypeError ===
    try:
        ex2_normalize_params_robust([42, 43], default_lr=1e-3)
    except TypeError:
        pass
    else:
        raise AssertionError('expected TypeError for int element')

    # === Feeds into a real torch.optim, both forms ===
    p5 = t.nn.Parameter(t.randn(3))
    p6 = t.nn.Parameter(t.randn(3))
    # Flat form
    out_flat = ex2_normalize_params_robust([p5, p6], default_lr=1e-2)
    opt_flat = t.optim.SGD(out_flat)
    # Group form with one missing-lr group
    out_dict = ex2_normalize_params_robust([{'params': [p5, p6]}], default_lr=1e-2)
    opt_dict = t.optim.SGD(out_dict)
    assert opt_flat.param_groups[0]['lr'] == 1e-2
    assert opt_dict.param_groups[0]['lr'] == 1e-2
    _dd_passed.add('ex2')
    print("ex2 ✓")

_test_ex2()

<details><summary>Solution</summary>

```python
def ex2_normalize_params_robust(params, default_lr):
    materialized = list(params)               # single-use safe
    if not materialized:
        raise ValueError('optimizer got an empty parameter list')
    first = materialized[0]
    if isinstance(first, t.Tensor):           # catches nn.Parameter too
        return [{'params': materialized, 'lr': default_lr}]
    if isinstance(first, dict):
        out = []
        for group in materialized:
            g = dict(group)                    # shallow copy — no caller mutation
            if 'lr' not in g:
                g['lr'] = default_lr
            out.append(g)
        return out
    raise TypeError(
        f'params must be an iterable of Tensors or dicts, '
        f'got first element of type {type(first).__name__}'
    )
```

**`isinstance` vs `type(...) is`.** `isinstance(first, t.Tensor)` returns True for `nn.Parameter` (a subclass). `type(first) is t.Tensor` would return False — a real bug, since every call site that passes `module.parameters()` would route through the dict branch and crash. This is the #1 polymorphism footgun in homemade optimizer code.

**Why `list(params)` BEFORE peeking.** Two reasons: (1) generators are single-use, and (2) some iterables (e.g. a one-shot `chain()`) advance internally when you call `iter()` on them. `list(...)` is the only safe primitive that gives you both 'first element' and 'all elements'.

**`dict(group)` not in-place mutation.** Mutating the caller's group dicts (`group['lr'] = default_lr`) is a silent side effect that breaks any code that introspects the dicts post-construction. PyTorch's own `Optimizer.__init__` shallow-copies each group for exactly this reason — match the source.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()